# A13 · 价值出价与预算、ROAS约束

先修：公用31–34；A12（22章）。

证据类型：**明确构造机制实验（非真实广告日志）**。本章不声称工业系统复现或线上收益。

学习顺序：先预测 → 只改一个变量 → 运行 → 解释结果。

## 问题与输入输出契约

风险中性、独立二价且无预算时，价值报价b=pCTCVR×conversion_value是每曝光价值。目标CPA出价b=target_CPA×pCTCVR，目标ROAS约束报价b=value/target_ROAS。它们依赖概率可靠性和计费口径；一价最优报价还依赖赢标概率，不能直接照搬。预算约束使请求之间耦合，需要机会成本或对偶价格。

运行前预测：本章只改变的变量会使哪一项结果变化？哪些不变量必须保持？

In [1]:
from pathlib import Path
import sys
root=Path.cwd()
if not (root/'reclab').exists(): root=root.parent
if str(root) not in sys.path: sys.path.insert(0,str(root))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from reclab import ads_curriculum as ac

## 运行单变量对照

输入是源码中可见的小型构造数据，输出单位和约束按上文定义。运行断言核验数学与账本。

In [2]:
rows=ac.constrained_bidding();display(pd.DataFrame(rows))
assert all(r['spend']<=4 for r in rows)
assert rows[1]['expected_roas']>=2
print('更紧预算',ac.constrained_bidding(budget=2))

,policy,spend,expected_value,expected_roas,win
0,value,3.6,7.0,1.944444,"[True, True, True, False, True, False]"
1,roas_cap,3.5,9.0,2.571429,"[False, True, False, False, False, True]"


更紧预算 [{'policy': 'value', 'spend': np.float64(1.9), 'expected_value': np.float64(4.5), 'expected_roas': np.float64(2.368421052631579), 'win': [True, True, False, False, False, False]}, {'policy': 'roas_cap', 'spend': np.float64(1.5), 'expected_value': np.float64(4.0), 'expected_roas': np.float64(2.6666666666666665), 'win': [False, True, False, False, False, False]}]


## 解释与局限

固定同一构造市场，只改变价值报价到ROAS阈值报价。即使期望ROAS达标，实际有限样本ROAS仍可能偏离。硬预算截断不等于全局价值最优；需结合下一单元反馈与到达次序讨论。

## 查看源码，再做练习

枚举全部6个请求的预算可行子集，得到离线knapsack上界，并检查两策略与上界差距。

In [3]:
import inspect
print(inspect.getsource(ac.constrained_bidding))

def constrained_bidding(budget=4., target_roas=2.):
    # Known constructed pCTCVR and conversion values; prices independent of bid.
    p=np.array([.01,.08,.02,.12,.03,.1]); value=p*50
    price=np.array([.4,1.5,.7,2.8,1.,2.])
    out=[]
    for name, bids in [('value',value),('roas_cap',value/target_roas)]:
        spent=reward=0.; decisions=[]
        for v,c,b in zip(value,price,bids):
            win=b>=c and spent+c<=budget+1e-12
            if win: spent+=c;reward+=v
            assert spent<=budget+1e-12
            decisions.append(bool(win))
        out.append(dict(policy=name,spend=spent,expected_value=reward,expected_roas=reward/spent if spent else None,win=decisions))
    return out



## 来源与下一步

A-RTB 71–91页。这是Lab自行设计的小实验，不复制讲义全文。

[GitHub中文单章](https://github.com/SocratesClub/ccrbook/blob/6dbd5650f2adc6e257e5bd5d156bc50a72560ee9/slides/Chapter%208%20%E8%AE%A1%E7%AE%97%E5%B9%BF%E5%91%8A.pptx) · [作者RTB讲义](https://wnzhang.net/slides/cikm-16-rtb-full.pdf)。中文PPT不是现代广告工程全套课；不将历史平台机制描述当作当前市场事实。

完成练习后回到广告学习路径，继续下一语义单元。